In [1]:

import cv2
import numpy as np

image = cv2.imread("input.jpg")

if image is None:
    raise FileNotFoundError("Cannot find input.jpg")

points = []

def select_points(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 4:
        points.append([x, y])
        print(f"Selected point: ({x}, {y})")

cv2.namedWindow("Select 4 Corners")
cv2.setMouseCallback("Select 4 Corners", select_points)

while True:
    display = image.copy()

    for i, p in enumerate(points):
        cv2.circle(display, tuple(p), 7, (0, 0, 255), -1)
        cv2.putText(
            display, str(i + 1),
            (p[0] + 8, p[1] - 8),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7,
            (0, 255, 0), 2
        )

    cv2.imshow("Select 4 Corners", display)
    key = cv2.waitKey(20) & 0xFF

    if key == ord('r'):
        points.clear()
    elif key == ord('q') and len(points) == 4:
        break

cv2.destroyAllWindows()

# Automatically order corners:
# top-left, top-right, bottom-right, bottom-left
pts = np.array(points, dtype=np.float32)
rect = np.zeros((4, 2), dtype=np.float32)

s = pts.sum(axis=1)
diff = np.diff(pts, axis=1).ravel()

rect[0] = pts[np.argmin(s)]      # Top-left
rect[2] = pts[np.argmax(s)]      # Bottom-right
rect[1] = pts[np.argmin(diff)]   # Top-right
rect[3] = pts[np.argmax(diff)]   # Bottom-left

# Calculate output dimensions from the selected quadrilateral
tl, tr, br, bl = rect

width = int(max(
    np.linalg.norm(br - bl),
    np.linalg.norm(tr - tl)
))
height = int(max(
    np.linalg.norm(tr - br),
    np.linalg.norm(tl - bl)
))

if width < 2 or height < 2:
    raise ValueError("Invalid corners selected.")

dst = np.float32([
    [0, 0],
    [width - 1, 0],
    [width - 1, height - 1],
    [0, height - 1]
])

H = cv2.getPerspectiveTransform(rect, dst)

result = cv2.warpPerspective(image, H, (width, height))

cv2.imwrite("rectified.jpg", result)

cv2.imshow("Original Image", image)
cv2.imshow("Rectified Image", result)

print("Saved rectified.jpg")
cv2.waitKey(0)
cv2.destroyAllWindows()


Selected point: (14, 102)
Selected point: (95, 229)
Selected point: (140, 46)
Selected point: (185, 151)
Saved rectified.jpg
